In [13]:
import pandas as pd
import os
import platform
import json
# from docx import Document
import numpy as np
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import matplotlib.pyplot as plt
import seaborn as sns

if platform.system() == 'Linux':
    Dir_github = r'/work/xy6/Project/ABC/github/' # HPC
else:
    Dir_github = r'X:\PROJECTS\ABC@UofSC\3-Experiments\NSF_valence\github' # Local computer
repo = 'ABC_valence'
pipeline = 'makeTimings'
Dir_working = os.path.join(Dir_github,repo,'scripts',pipeline)
Dir_ratings = os.path.join(Dir_github,'derivatives','behavioral','ratings')


In [12]:
dfs_ratings = []
# get gpt ratings
List_model = ['gpt-4-1106-preview','gpt-4o-2024-08-06','gpt-4o-2024-11-20']
for model in List_model:
    df = pd.read_csv(os.path.join(Dir_ratings,'gpt',model,f"rating_gpt_{model}.csv"))
    dfs_ratings.append(df)

# get human ratings
List_qualtrics = ['ABCNaturalisticNarrativeAudio_sona_A']
for project in List_qualtrics:
    df = pd.read_csv(os.path.join(Dir_ratings,'human','raw',f"rating_human_{project}.csv"))
    dfs_ratings.append(df)

dfs_ratings = pd.concat(dfs_ratings,ignore_index=True)
dfs_ratings

,segment,reason,story,idx,audio,segment_raw,onset,offset,words,idx_raw,...,V,A,firstlast,firstword,lastword,flag,iteration,rater,subid,subid_idx
0,"Long, long ago",positioning element,AgedMother,1,Jacob_AgedMother,"Long, long ago",5.200000,6.36,Long|long|ago,0,...,0.00000,2.00000,Long ago,Long,ago,11.0,0,gpt-4-1106-preview,NaN,NaN
1,there lived at the foot of the mountain,positioning element,AgedMother,2,Jacob_AgedMother,there lived at the foot of the mountain,6.650000,8.29,there|lived|at|the|foot|of|the|mountain,1,...,1.00000,3.00000,there mountain,there,mountain,11.0,0,gpt-4-1106-preview,NaN,NaN
2,"a poor farmer and his aged, widowed mother",>maximal segment,AgedMother,3,Jacob_AgedMother,"a poor farmer and his aged, widowed mother.",8.309999,11.07,a|poor|farmer|and|his|aged|widowed|mother,2,...,-1.00000,4.00000,a mother,a,mother,11.0,0,gpt-4-1106-preview,NaN,NaN
3,They owned a bit of land,-,AgedMother,4,Jacob_AgedMother,They owned a bit of land,12.080000,13.30,They|owned|a|bit|of|land,3,...,1.00000,3.00000,They land,They,land,11.0,0,gpt-4-1106-preview,NaN,NaN
4,which supplied them with food,relative clause,AgedMother,5,Jacob_AgedMother,"which supplied them with food,",13.309999,14.79,which|supplied|them|with|food,4,...,2.00000,3.00000,which food,which,food,11.0,0,gpt-4-1106-preview,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
72727,"and she's got like 50 thousand ""likes",>maximal segment,Koko,89,WavenetF_Koko,"and she's got like 50 thousand ""likes.""",338.049999,341.09,and|she's|got|like|50|thousand|likes,89,...,0.82500,5.11875,NaN,NaN,NaN,NaN,10,human,R_68AhNRZHfKB94ch,48617.0
72728,Some may deny,-,Koko,90,WavenetF_Koko,Some may deny,342.859999,344.01,Some|may|deny,90,...,-2.71875,6.22500,NaN,NaN,NaN,NaN,10,human,R_68AhNRZHfKB94ch,48617.0
72729,that she knows sign language,complementizer,Koko,91,WavenetF_Koko,"that she knows sign language,",344.010000,346.03,that|she|knows|sign|language,91,...,1.27500,5.70000,NaN,NaN,NaN,NaN,10,human,R_68AhNRZHfKB94ch,48617.0
72730,but nobody says,conjunction,Koko,92,WavenetF_Koko,but nobody says,346.090000,347.41,but|nobody|says,92,...,-1.74375,5.21250,NaN,NaN,NaN,NaN,10,human,R_68AhNRZHfKB94ch,48617.0


In [21]:
dfs_ratings_mean = dfs_ratings[['rater','story','iteration','idx_raw','V','A','segment','onset','offset']].groupby(['rater','story','idx_raw','segment']).mean().reset_index()
dfs_ratings_mean

,rater,story,idx_raw,segment,iteration,V,A,onset,offset
0,gpt-4-1106-preview,AgedMother,0,"Long, long ago",4.555556,0.000000,2.000000,5.200000,6.36
1,gpt-4-1106-preview,AgedMother,1,there lived at the foot of the mountain,4.555556,1.000000,3.000000,6.650000,8.29
2,gpt-4-1106-preview,AgedMother,2,"a poor farmer and his aged, widowed mother",4.555556,-1.000000,4.000000,8.309999,11.07
3,gpt-4-1106-preview,AgedMother,3,They owned a bit of land,4.555556,1.000000,3.000000,12.080000,13.30
4,gpt-4-1106-preview,AgedMother,4,which supplied them with food,4.555556,2.000000,3.000000,13.309999,14.79
...,...,...,...,...,...,...,...,...,...
9233,human,Koko,89,"and she's got like 50 thousand ""likes",6.000000,1.391667,4.456250,338.049999,341.09
9234,human,Koko,90,Some may deny,6.000000,-0.141667,3.835417,342.859999,344.01
9235,human,Koko,91,that she knows sign language,6.000000,-0.485417,3.685417,344.010000,346.03
9236,human,Koko,92,but nobody says,6.000000,0.158333,2.735417,346.090000,347.41


In [51]:
df_plot = dfs_ratings_mean[['story','idx_raw','segment','V','rater']]
df_plot = df_plot.pivot(index=['story','idx_raw','segment'],columns='rater',values='V').reset_index().dropna()
List_vars = [x for x in df_plot.columns if 'gpt' in x or 'human' in x]
dfs_corr = []
for story in df_plot['story'].unique():
    # df_plot.dropna()[List_vars].corr()
    df_tmp = df_plot.loc[df_plot['story']==story,List_vars].corr()
    df_tmp['story'] = story
    dfs_corr.append(df_tmp)
dfs_corr = pd.concat(dfs_corr)
dfs_corr

rater,gpt-4-1106-preview,gpt-4o-2024-08-06,gpt-4o-2024-11-20,human,story
rater,,,,,
gpt-4-1106-preview,1.000000,0.745553,0.351530,0.582323,Aqua
gpt-4o-2024-08-06,0.745553,1.000000,0.332721,0.676155,Aqua
gpt-4o-2024-11-20,0.351530,0.332721,1.000000,0.218163,Aqua
human,0.582323,0.676155,0.218163,1.000000,Aqua
gpt-4-1106-preview,1.000000,0.545699,0.410098,0.681092,FireSelected
gpt-4o-2024-08-06,0.545699,1.000000,0.932402,0.860832,FireSelected
gpt-4o-2024-11-20,0.410098,0.932402,1.000000,0.821820,FireSelected
human,0.681092,0.860832,0.821820,1.000000,FireSelected
gpt-4-1106-preview,1.000000,0.492763,0.286479,0.375549,Koko
